# K-SPADA Data Preparation

This tutorial prepares training, validation, and query communication-graph files from a generic spatial transcriptomics AnnData object.

## Workflow
1. Define paths and sample splits.
2. Validate the required AnnData fields.
3. Separate reference and query samples.
4. Build pathway features and communication graphs.

In [ ]:
from pathlib import Path
import anndata as ad
import KSPADA

PROJECT_ROOT = Path("/your_path/K-SPADA")
DATA_DIR = Path("/your_path/example_dataset")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_preparation"
SOURCE_H5AD = DATA_DIR / "source_data.h5ad"

TRAIN_SAMPLES = ["train_sample_1", "train_sample_2"]
VALIDATION_SAMPLES = ["validation_sample"]
QUERY_SAMPLES = ["query_sample"]
COMMUNICATION_GRAPH_NEIGHBORS = 2
FORCE_OVERWRITE = True

## Step 1 — Validate the source data

The source AnnData object must contain spatial coordinates, sample identifiers, and training labels.

In [ ]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
required_columns = {"x", "y", "batch", "Celltype"}
missing_columns = required_columns.difference(source_adata.obs.columns)
if missing_columns:
    raise KeyError(f"Missing required obs columns: {sorted(missing_columns)}")

source_adata.obs["batch"] = source_adata.obs["batch"].astype(str)
requested_samples = set(TRAIN_SAMPLES + VALIDATION_SAMPLES + QUERY_SAMPLES)
missing_samples = requested_samples.difference(source_adata.obs["batch"].unique())
if missing_samples:
    raise ValueError(f"Missing requested samples: {sorted(missing_samples)}")

## Step 2 — Split reference and query data

In [ ]:
train_valid_adata = source_adata[
    source_adata.obs["batch"].isin(TRAIN_SAMPLES + VALIDATION_SAMPLES)
].copy()
query_adata = source_adata[source_adata.obs["batch"].isin(QUERY_SAMPLES)].copy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_valid_source = OUTPUT_DIR / "reference_source.h5ad"
query_source = OUTPUT_DIR / "query_source.h5ad"
KSPADA.write_adata(train_valid_adata, train_valid_source, force=FORCE_OVERWRITE)
KSPADA.write_adata(query_adata, query_source, force=FORCE_OVERWRITE)

## Step 3 — Build pathway features and communication graphs

In [ ]:
prepared_paths = KSPADA.prepare_data_workflow(
    train_h5ad=train_valid_source,
    query_h5ad=query_source,
    output_dir=OUTPUT_DIR,
    train_batches=TRAIN_SAMPLES,
    valid_batches=VALIDATION_SAMPLES,
    query_batches=QUERY_SAMPLES,
    cgraph_k=COMMUNICATION_GRAPH_NEIGHBORS,
    train_output_name="reference_data",
    query_output_name="query_data",
    force=FORCE_OVERWRITE,
)

## Notes

- Keep biological samples separate when assigning training, validation, and query data.
- Replace the sample names with values present in `adata.obs["batch"]`.